In [ ]:
import arcpy
import os
from datetime import datetime

# --------------------------------------------------------------------------
# USER INPUTS
# --------------------------------------------------------------------------
row_input_fc = r"C:\Exclusion_Analysis_Phase3\Input_data\UpdatedROWs_010926\CONUS_Revised_NHS_ROW.gdb\CONUS_Revised"
exclusion_raster = r"C:\Exclusion_Analysis_Phase3\scratch.gdb\Exclusion_90m_buffer_extract_INT_con_RastertoPoly"
fishnet_fc = r"C:\Exclusion_Analysis_Phase3\Exclusion_Analysis_Phase3.gdb\conus_fishnet_500"
output_gdb = r"C:\Exclusion_Analysis_Phase3\output.gdb"
final_output_fc = os.path.join(output_gdb, "CONUS_Revised_ROW_90m_FastChunked")

chunk_size = 50000  # number of ROW features per chunk (adjust if needed)

# --------------------------------------------------------------------------
# ENVIRONMENT
# --------------------------------------------------------------------------
arcpy.env.overwriteOutput = True

# --------------------------------------------------------------------------
# LOGGING
# --------------------------------------------------------------------------
log_file = os.path.join(output_gdb, f"row_clip_chunked_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")
def log(msg):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] {msg}")
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(f"[{timestamp}] {msg}\n")

log("=== SCRIPT START ===")
log(f"ROW Input: {row_input_fc}")
log(f"Exclusion Raster: {exclusion_raster}")
log(f"Fishnet: {fishnet_fc}")
log(f"Output GDB: {output_gdb}")

# --------------------------------------------------------------------------
# Make feature layers
# --------------------------------------------------------------------------
row_layer = "row_layer"
tiles_layer = "tiles_layer"

if arcpy.Exists(row_layer):
    arcpy.management.Delete(row_layer)
if arcpy.Exists(tiles_layer):
    arcpy.management.Delete(tiles_layer)

arcpy.management.MakeFeatureLayer(row_input_fc, row_layer)
arcpy.management.MakeFeatureLayer(fishnet_fc, tiles_layer)

# --------------------------------------------------------------------------
# Select tiles that intersect ROW polygons
# --------------------------------------------------------------------------
log("Selecting tiles that intersect ROW polygons...")
arcpy.management.SelectLayerByLocation(tiles_layer, "INTERSECT", row_layer)
tile_oids = [row[0] for row in arcpy.da.SearchCursor(tiles_layer, arcpy.Describe(tiles_layer).OIDFieldName)]
total_tiles = len(tile_oids)
log(f"Tiles to process: {total_tiles}")

# --------------------------------------------------------------------------
# Process each tile with chunking
# --------------------------------------------------------------------------
processed_tiles = []
start_time = datetime.now()

for i, oid in enumerate(tile_oids, start=1):
    log(f"START Tile {i} (OID {oid})")

    # Select tile
    where_clause = f"{arcpy.AddFieldDelimiters(tiles_layer, arcpy.Describe(tiles_layer).OIDFieldName)} = {oid}"
    arcpy.management.SelectLayerByAttribute(tiles_layer, "NEW_SELECTION", where_clause)

    # Clip ROW polygons by tile
    tile_clip_fc = os.path.join(output_gdb, f"clip_tile_{oid}")
    arcpy.analysis.Clip(row_layer, tiles_layer, tile_clip_fc)

    # Skip empty tiles
    count = int(arcpy.management.GetCount(tile_clip_fc)[0])
    if count == 0:
        log(f"  -> Tile {oid} empty, skipping")
        arcpy.management.Delete(tile_clip_fc)
        arcpy.management.SelectLayerByAttribute(tiles_layer, "CLEAR_SELECTION")
        continue

    # Split tile_clip into chunks
    oid_field = arcpy.Describe(tile_clip_fc).OIDFieldName
    all_oids = [row[0] for row in arcpy.da.SearchCursor(tile_clip_fc, [oid_field])]

    chunk_outputs = []
    for start in range(0, len(all_oids), chunk_size):
        chunk_oids = all_oids[start:start+chunk_size]
        oid_list_str = ",".join(map(str, chunk_oids))
        chunk_where = f"{arcpy.AddFieldDelimiters(tile_clip_fc, oid_field)} IN ({oid_list_str})"
        
        chunk_layer = f"chunk_layer_{oid}_{start}"
        arcpy.management.MakeFeatureLayer(tile_clip_fc, chunk_layer, chunk_where)

        # Extract by raster mask
        chunk_extract_fc = os.path.join(output_gdb, f"extract_tile_{oid}_{start}")
        arcpy.analysis.Clip(chunk_layer, exclusion_raster, chunk_extract_fc)

        # Dissolve within chunk
        chunk_dissolve_fc = os.path.join(output_gdb, f"dissolve_tile_{oid}_{start}")
        arcpy.analysis.PairwiseDissolve(chunk_extract_fc, chunk_dissolve_fc)
        chunk_outputs.append(chunk_dissolve_fc)

        # Cleanup
        arcpy.management.Delete(chunk_layer)
        arcpy.management.Delete(chunk_extract_fc)

    # Merge all chunks for this tile
    if chunk_outputs:
        tile_final_fc = os.path.join(output_gdb, f"tile_final_{oid}")
        arcpy.management.Merge(chunk_outputs, tile_final_fc)
        processed_tiles.append(tile_final_fc)

        # Cleanup chunk dissolves
        for c in chunk_outputs:
            arcpy.management.Delete(c)

    # Cleanup tile_clip
    arcpy.management.Delete(tile_clip_fc)
    arcpy.management.SelectLayerByAttribute(tiles_layer, "CLEAR_SELECTION")

    # ETA
    elapsed = datetime.now() - start_time
    avg_time = elapsed / i
    remaining = avg_time * (total_tiles - i)
    eta = datetime.now() + remaining
    log(f"  -> Tile {oid} processed")
    log(f"  -> Progress: {i}/{total_tiles} tiles ({(i/total_tiles)*100:.1f}%) | ETA: {eta.strftime('%Y-%m-%d %H:%M:%S')}")

# --------------------------------------------------------------------------
# Merge all tile outputs
# --------------------------------------------------------------------------
if processed_tiles:
    log("Merging all tiles...")
    merged_fc = os.path.join(output_gdb, "merged_tiles_chunked")
    arcpy.management.Merge(processed_tiles, merged_fc)

    log("Final dissolve...")
    arcpy.analysis.PairwiseDissolve(merged_fc, final_output_fc)
    log(f"Final output: {final_output_fc}")

    # Cleanup intermediate tile files
    for t in processed_tiles:
        arcpy.management.Delete(t)
else:
    log("No tiles processed. Check inputs or tile size.")

log("=== SCRIPT END ===")
